# FLUX.2 Klein 9B — Garment T-pose Conversion + Gradio API

Auto-generated notebook for recipe: **flux2-tpose**


# FLUX.2 Klein 9B — Garment T-pose Conversion

단일 의류 사진을 **T-pose 정면 이미지**로 변환합니다.

- **Model**: `black-forest-labs/FLUX.2-klein-9B` (9B params, 4-step distilled)
- **Input**: 의류 사진 (평면촬영, 행거샷, 착용샷)
- **Output**: T-pose 의류 이미지 (흰배경, 1024x1024)
- **API**: Gradio 엔드포인트로 외부 호출 가능

---


## A) GPU Check


In [ ]:
#@title A) GPU & VRAM Check { run: "auto" }
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Change runtime: Runtime > Change runtime type > GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 29:
    print("WARNING: VRAM < 29GB. cpu_offload will be used but may be slow.")
elif vram_gb >= 40:
    print(f"OK: {vram_gb:.0f}GB — full GPU mode (no offload needed)")
else:
    print("OK: Sufficient VRAM for FLUX.2 Klein 9B")


## B) Install Dependencies


In [ ]:
#@title B) Install Dependencies { run: "auto" }
import importlib, subprocess, sys, os

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# --- 1. Probe Flux2KleinPipeline ---
_need_restart = False
try:
    from diffusers import Flux2KleinPipeline
    print(f"Flux2KleinPipeline found in diffusers {importlib.metadata.version('diffusers')}")
except ImportError:
    print("Flux2KleinPipeline not in current diffusers — installing git HEAD...")
    pip_install("git+https://github.com/huggingface/diffusers.git")
    _need_restart = True

# --- 2. gradio + deps ---
for pkg in ["gradio>=5.0", "accelerate>=1.12.0",
             "sentencepiece", "protobuf", "safetensors"]:
    pip_install(pkg)
print("Additional dependencies installed")

# --- 3. Restart or verify ---
if _need_restart:
    print("\nRestarting runtime to pick up new diffusers...")
    print(">>> After restart, re-run from this cell (B). It will skip install and verify.")
    import signal
    os.kill(os.getpid(), signal.SIGKILL)
else:
    import torch, transformers, accelerate
    print(f"\ntorch={torch.__version__}, CUDA={torch.version.cuda}")
    print(f"transformers={transformers.__version__}")
    print(f"accelerate={accelerate.__version__}")
    print(f"diffusers={importlib.metadata.version('diffusers')}")
    print("\nAll ready.")


## C) HuggingFace Authentication

FLUX.2 Klein 9B는 gated model — HF 토큰이 필요합니다.


In [ ]:
#@title C) HuggingFace Login { run: "auto" }
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Colab secret 'HF_TOKEN'")
except Exception:
    print("HF_TOKEN not found in Colab secrets.")
    print("If download fails, add your token: Colab sidebar > Secrets > HF_TOKEN")
    print("Or run: huggingface_hub.login()")


## D) Load FLUX.2 Klein 9B


In [ ]:
#@title D) Load Model { run: "auto" }
import torch
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-9B"
DTYPE = torch.bfloat16

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Loading {MODEL_ID} ...")
pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)

if vram_gb >= 40:
    pipe.to("cuda")
    print(f"Full GPU mode (VRAM: {vram_gb:.0f}GB)")
else:
    pipe.enable_model_cpu_offload()
    print(f"CPU offload mode (VRAM: {vram_gb:.0f}GB)")

vram_after = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded. VRAM used: {vram_after:.2f} GB")
print("Ready for inference.")


## E) Preprocessing — Center on Wide Canvas


In [ ]:
#@title E) Center on Wide Canvas (1536x1024)
from PIL import Image

def preprocess_garment(img: Image.Image, target_w: int = 1536, target_h: int = 1024) -> Image.Image:
    """Center the garment on a wide white canvas for T-pose sleeve room.
    70% fill to leave horizontal space for sleeve rotation."""
    img = img.convert("RGB")
    w, h = img.size

    pad_px = int(min(target_w, target_h) * 0.05)
    avail_w = target_w - 2 * pad_px
    avail_h = target_h - 2 * pad_px
    scale = min(avail_w / w, avail_h / h) * 0.7
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = img.resize((new_w, new_h), Image.LANCZOS)

    canvas = Image.new("RGB", (target_w, target_h), (255, 255, 255))
    offset_x = (target_w - new_w) // 2
    offset_y = (target_h - new_h) // 2
    canvas.paste(resized, (offset_x, offset_y))

    return canvas

print("Preprocessing function loaded.")


## F) T-pose Prompt Builder

프롬프트 엔지니어링 전략:
- **Subject + Action + Style + Context** 구조
- 100단어 이내 (Klein 최적)
- T-pose/flat-lay를 맨 앞에 (decoder-only causal attention)
- No negative prompts (FLUX.2 미지원)


In [ ]:
#@title F) Prompt Builder

def build_tpose_prompt() -> str:
    """Build the T-pose conversion prompt (edit-instruction style for Klein 9B).
    No garment type specified — the reference image is the sole source of truth."""
    prompt = (
        "Reposition the garment from the input photo into a flat-lay T-pose layout. "
        "Rotate the sleeves outward around the shoulder seams so they extend horizontally, "
        "symmetric left and right. "
        "Keep the EXACT same sleeve length as the input — do NOT lengthen or shorten them. "
        "A short-sleeve garment must remain short-sleeve. A long-sleeve garment must remain long-sleeve. "
        "Preserve all garment identity exactly: color, fabric texture, stitching, seams, "
        "hood shape, pockets, graphic prints, and overall proportions. "
        "Do not add or remove any part of the garment. "
        "Garment only, no person, no mannequin, no hanger, clean white background, fully visible."
    )
    return prompt

# Preview
sample_prompt = build_tpose_prompt()
word_count = len(sample_prompt.split())
print(f"Sample prompt ({word_count} words):")
print(sample_prompt)


## G) Single Inference Test

의류 이미지를 업로드하고 T-pose 변환을 테스트합니다.


In [ ]:
#@title G) Upload & Convert to T-pose
import torch
from PIL import Image
from google.colab import files
import time

#@markdown ### Settings
GUIDANCE = 4.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
NUM_STEPS = 4  #@param {type:"slider", min:4, max:50, step:1}
SEED = 42  #@param {type:"integer"}
OUT_W = 1536  #@param [1024, 1280, 1536] {type:"raw"}
OUT_H = 1024  #@param [768, 1024] {type:"raw"}

# --- Upload ---
print("Upload a garment image:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
garment_img = Image.open(filename).convert("RGB")
print(f"Uploaded: {filename} ({garment_img.size})")

# --- Preprocess (center on wide canvas) ---
print("Centering on wide canvas...")
preprocessed = preprocess_garment(garment_img, OUT_W, OUT_H)
display(preprocessed)
print(f"Preprocessed: {preprocessed.size}")

# --- Build prompt ---
prompt = build_tpose_prompt()
print(f"\nPrompt: {prompt}")

# --- Inference (in-context conditioning, BFL Space pattern) ---
print(f"\nGenerating T-pose ({OUT_W}x{OUT_H}, steps={NUM_STEPS}, guidance={GUIDANCE}, seed={SEED})...")
generator = torch.Generator(device="cuda").manual_seed(SEED)

kwargs = dict(
    prompt=prompt,
    image=[preprocessed],
    height=OUT_H,
    width=OUT_W,
    guidance_scale=GUIDANCE,
    num_inference_steps=NUM_STEPS,
    generator=generator,
)

t0 = time.time()
result = pipe(**kwargs).images[0]
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
display(result)

# --- Save ---
out_path = f"tpose_{filename}"
result.save(out_path)
print(f"Saved: {out_path}")


## H) Gradio App + API Endpoint

`share=True`로 공개 URL이 생성됩니다.
Gradio Client로 API 호출 가능:

```python
from gradio_client import Client
client = Client("https://xxxxx.gradio.live")
result = client.predict(
    image="garment.jpg",
    seed=42,
    api_name="/convert"
)
```


In [ ]:
#@title H) Launch Gradio App + API { run: "auto" }
import gradio as gr
import torch
from PIL import Image
import time
import numpy as np

def convert_to_tpose(
    image: Image.Image,
    guidance: float = 4.0,
    num_steps: int = 4,
    seed: int = 42,
    out_w: int = 1536,
    out_h: int = 1024,
) -> tuple[Image.Image, Image.Image, str]:
    """Convert garment image to T-pose. Returns (preprocessed, result, info)."""
    if image is None:
        raise gr.Error("Please upload a garment image.")

    image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    image = image.convert("RGB")

    # Center on wide canvas
    preprocessed = preprocess_garment(image, out_w, out_h)

    # Build prompt
    prompt = build_tpose_prompt()

    # Inference (BFL Space pattern)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    kwargs = dict(
        prompt=prompt,
        image=[preprocessed],
        height=out_h,
        width=out_w,
        guidance_scale=guidance,
        num_inference_steps=num_steps,
        generator=generator,
    )
    t0 = time.time()
    result = pipe(**kwargs).images[0]
    elapsed = time.time() - t0

    info = f"Time: {elapsed:.1f}s | {out_w}x{out_h} | Guidance: {guidance} | Steps: {num_steps} | Seed: {seed}"
    return preprocessed, result, info

# --- Build Gradio UI ---
with gr.Blocks(title="Garment T-pose Converter") as demo:
    gr.Markdown("# Garment T-pose Converter\nUpload a garment photo to convert it to T-pose flat-lay format.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input Garment", type="pil")
            guidance = gr.Slider(1.0, 10.0, value=4.0, step=0.5, label="Guidance Scale")
            num_steps = gr.Slider(4, 50, value=4, step=1, label="Inference Steps")
            seed = gr.Number(value=42, label="Seed", precision=0)
            out_w = gr.Radio([1024, 1280, 1536], value=1536, label="Output Width")
            out_h = gr.Radio([768, 1024], value=1024, label="Output Height")
            btn = gr.Button("Convert to T-pose", variant="primary")

        with gr.Column():
            preprocessed_output = gr.Image(label="Preprocessed (bg removed)")
            result_output = gr.Image(label="T-pose Result")
            info_output = gr.Textbox(label="Info")

    btn.click(
        fn=convert_to_tpose,
        inputs=[input_image, guidance, num_steps, seed, out_w, out_h],
        outputs=[preprocessed_output, result_output, info_output],
        api_name="convert",
    )

# --- Launch ---
print("Launching Gradio app with public URL...")
demo.launch(share=True, quiet=False)
